<a href="https://colab.research.google.com/github/rcodo/eth-fdd-fs26-we2-explainable-AI-LLM-Optimization/blob/my-sol/exercises/01_nn_language_model_students.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Block 1 — Re-Intro to Neural Networks: Build a Character-Level Language Model

> **The big idea.** Every large language model, underneath, does one thing: read the text so far and
> **predict the next token**. Today we build that core from scratch — a neural network that reads the
> last few *characters* of a name and predicts the next one. Train it, and it will invent brand-new
> names it has never seen. No transformers, no attention — just the neural-network fundamentals every
> model is built on, in **idiomatic PyTorch** (`nn.Module`, an optimizer, a `DataLoader`).

**How this notebook works**

- Short idea, then small hands-on tasks marked **🎯** for you to fill in (usually one line).
- Every task is followed by a quick **check** (an `assert` or a plot) so you know it worked.
- Runs top-to-bottom on **Google Colab** — `Runtime → Run all`. With a GPU runtime it trains in seconds.
- **Mixed-audience note:** cells labelled *Run Without Modification* are plumbing — just run them.
  Sections marked **⏭️ optional** are bonus; skip them if you are short on time. Nothing here can
  "break": if a task errors, it is just telling you that one line is not filled in yet.

**What you will build**

1. A character **vocabulary** and the `text → integers` mapping (tokenization).
2. A **model** — an `nn.Module` with an embedding layer and two linear layers.
3. A **loss** (`nn.CrossEntropyLoss`) and the standard PyTorch **training loop** over a `DataLoader`.
4. **Generation**: sample the model to invent new names, one character at a time.
5. A **peek inside** the trained model's learned character map.

**Where this is going.** By the end you will have built **next-token prediction — the core of every
LLM**. A real LLM keeps this exact recipe but swaps our little MLP for a **transformer** and characters
for sub-word tokens. Prying that transformer open is exactly Saturday's **mechanistic-interpretability**
session.

### First, the two ideas from the lecture

The talk just showed you what a neural network really is, in two pictures. This whole notebook is those
two ideas turned into code — we simply point them at *text*.

1. **A neural network is neurons stacked to build a function.** Each neuron is a tiny, simple unit; stack
   enough of them and, together, they can bend into *any* shape — approximating whatever function the
   data needs. That is the *"stack neurons → build a curve"* picture. In our model that stack is the
   **hidden layer** (Section 2): 256 neurons whose job is to shape context into good predictions.
2. **We train it by gradient descent.** We measure how wrong the network is (the **loss**), then walk
   every parameter a little **downhill** on that loss surface. Repeat thousands of times and the network
   rolls into a good setting. That is the *"ball rolling down the loss landscape"* picture, and it is
   exactly the training loop in Section 4.

Everything else here — turning characters into numbers, and sampling the trained network to invent
names — is just the *scaffolding* that lets those two ideas do something fun. Keep the two pictures in
mind; we call them out again when we reach them.

In [1]:
#@title <🔧 Imports, seed & device — Run Without Modification> {display-mode: "form"}
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("✅ ready — torch", torch.__version__, "| device:", device)

✅ ready — torch 2.11.0+cu128 | device: cuda


## 1. From text to numbers (tokenization)

Our dataset is a big list of **names**. We treat each name as a sequence of **characters** and teach
the model one simple game: *given the last few characters, guess the next one.*

We add a special boundary token `.` to mark where a name **starts and ends** (so the model can learn
which letters begin a name, and when to stop). With a context window of **N = 3** characters, the name
`emma` becomes these `(context → target)` pairs:

```
...  →  e
..e  →  m
.em  →  m
emm  →  a
mma  →  .   (end)
```

The next cells download ~32,000 real names (with an automatic offline fallback), build the character
**vocabulary**, and then turn every name into those training pairs.

In [2]:
#@title <📊 Load the names dataset (with offline fallback) — Run Without Modification> {display-mode: "form"}
import urllib.request

NAMES_URL = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"

FALLBACK_NAMES = """
emma olivia ava isabella sophia charlotte mia amelia harper evelyn abigail emily elizabeth
mila ella avery sofia camila aria scarlett victoria madison luna grace chloe penelope layla
riley zoey nora lily eleanor hannah lillian addison aubrey ellie stella natalie zoe leah hazel
violet aurora savannah audrey brooklyn bella claire skylar lucy paisley everly anna caroline
nova genesis emilia kennedy samantha maya willow kinsley naomi aaliyah elena sarah ariana
allison gabriella alice madelyn cora ruby eva serenity autumn adeline hailey gianna valentina
isla eliana quinn nevaeh ivy sadie piper lydia alexa josephine emery julia delilah arya vivian
kaylee sophie brielle madeline peyton rylee clara hadley melanie mackenzie reagan adalynn
liliana aubree jade katherine isabelle
liam noah oliver william elijah james benjamin lucas mason ethan alexander henry jacob michael
daniel logan jackson sebastian jack aiden owen samuel matthew joseph levi mateo david john
wyatt carter julian luke grayson isaac jayden theodore gabriel anthony dylan leo lincoln jaxon
asher christopher josiah andrew thomas joshua ezra hudson charles caleb isaiah ryan nathan
adrian christian maverick colton elias aaron eli landon jonathan nolan hunter cameron connor
santiago jeremiah ezekiel angel roman easton axel xavier jose jace jameson leonardo bryson
kayden miles sawyer jason declan weston micah ayden wesley luca everett max silas
""".split()

try:
    with urllib.request.urlopen(NAMES_URL, timeout=5) as resp:
        words = resp.read().decode("utf-8").splitlines()
    words = [w.strip().lower() for w in words if w.strip().isalpha()]
    if len(words) < 100:
        raise ValueError("download looked empty")
    source = "download"
except Exception as e:
    words = FALLBACK_NAMES
    source = f"offline fallback ({type(e).__name__})"

print(f"loaded {len(words)} names via {source}")
print("examples:", words[:8])
print("shortest / longest name length:", min(map(len, words)), "/", max(map(len, words)))

loaded 32033 names via download
examples: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
shortest / longest name length: 2 / 15


In [3]:
#@title <📊 Build the character vocabulary — Run Without Modification> {display-mode: "form"}
chars = sorted(set("".join(words)))     # every distinct character in the data (a-z)
stoi = {".": 0}                          # '.' is our start/end token -> id 0
for i, c in enumerate(chars):
    stoi[c] = i + 1                      # 'a'->1, 'b'->2, ... 'z'->26
itos = {i: c for c, i in stoi.items()}   # reverse map: id -> character
vocab_size = len(stoi)

print("vocab_size =", vocab_size)
print("stoi =", stoi)

vocab_size = 27
stoi = {'.': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26}


### 🎯 Task 1.1 — Turn text into integers

A neural network cannot read letters, only numbers. **Tokenization** is just the lookup that swaps
each character for its integer id using `stoi`. Fill in that mapping below.

In [4]:
def encode(word):
    # 🎯 TODO — map each character in `word` to its integer id using `stoi`
    return [stoi[c] for c in word]

def decode(ids):
    return "".join(itos[i] for i in ids)   # the inverse, provided for you

demo = encode("emma")
print("emma      ->", demo)
print("and back  ->", decode(demo))

assert demo == [5, 13, 13, 1], f"expected [5, 13, 13, 1] for 'emma', got {demo}"
assert decode(encode("noah")) == "noah", "decode(encode(x)) should return the original string"
print("✅ tokenization works — text is now integers")

emma      -> [5, 13, 13, 1]
and back  -> emma
✅ tokenization works — text is now integers


In [6]:
#@title <📊 Build the (context → next-character) training pairs — Run Without Modification> {display-mode: "form"}
BLOCK_SIZE = 3   # how many previous characters the model sees (the context window)

def build_dataset(word_list):
    X, Y = [], []
    for w in word_list:
        context = [0] * BLOCK_SIZE               # start context is '...' (all boundary tokens)
        for ch in w + ".":                       # walk the name, then the end token
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]         # slide the window forward
    return torch.tensor(X), torch.tensor(Y)

# shuffle, then split 90% train / 10% validation
perm = np.random.permutation(len(words))
split = int(0.9 * len(words))
train_words = [words[i] for i in perm[:split]]
val_words   = [words[i] for i in perm[split:]]

Xtr, Ytr   = build_dataset(train_words)   # kept on CPU; each batch is moved to `device` during training
Xval, Yval = build_dataset(val_words)

print("Xtr:", tuple(Xtr.shape), " Ytr:", tuple(Ytr.shape), " dtype:", Xtr.dtype)
print("\nfirst 5 (context -> target) pairs from the name:", train_words[0])
for i in range(5):
    print("   ", decode(Xtr[i].tolist()), "->", itos[Ytr[i].item()])

Xtr: (205425, 3)  Ytr: (205425,)  dtype: torch.int64

first 5 (context -> target) pairs from the name: leylanie
    ... -> l
    ..l -> e
    .le -> y
    ley -> l
    eyl -> a


> **💡 Insight.** That table is the *entire* dataset of a language model: a long list of
> *(what came before → what comes next)*. GPT-style models do exactly this, only with billions of
> sub-word tokens instead of a few hundred thousand characters.

## 2. The model — an `nn.Module`

In PyTorch you define a network as a class that subclasses **`nn.Module`**. Two methods matter:

- **`__init__`** — create the **layers** (each is itself a small `nn.Module` that holds trainable weights):
  - `nn.Embedding(vocab_size, emb_dim)` — a learnable **lookup table**: character id → a short vector.
  - `nn.Linear(in, out)` — a fully-connected layer (`x @ W.T + b`), weights created for you.
- **`forward(x)`** — the **data flow**: how an input batch turns into outputs.

Our network is deliberately small and see-through:

1. **Embed** each of the 3 context characters → `(batch, 3, 24)`
2. **Flatten** to 72 numbers per example → `(batch, 72)`  *(this concatenates the context)*
3. **Linear → tanh** hidden layer → `(batch, 256)`
4. **Linear** output layer → `(batch, 27)` — one **score (logit)** per possible next character

Each character is embedded into **`EMB_DIM = 24`** numbers — enough capacity for the model to make
good predictions. (In the optional "peek inside" at the end we compress those 24 numbers down to 2
with **PCA** so we can still plot them and see what the model figured out.)

### The MLP — the flow of information

Read it top to bottom: the 3 previous characters flow through the layers into a probability for each
possible next character.

```text
   "e"  "m"  "m"              the 3 previous characters (as ids)
      |
      v   embed: each character -> a 24-number vector
   (3 x 24)
      |
      v   flatten: glue the 3 vectors into one row
   (72,)
      |
      v   Linear (72 -> 256), then tanh
   (256,)                     hidden layer
      |
      v   Linear (256 -> 27)
   (27,)                      one score per possible next character
      |
      v   softmax
   P(next character)          27 probabilities that sum to 1
```

> **💡 Lecture idea #1, in code.** That **hidden layer** — `Linear (72 → 256)` then `tanh` — *is* the
> lecture's *"stack neurons to build a function."* Each of the 256 units is one neuron; together they
> bend into whatever shape best maps a 3-character context to next-character scores. Swap `tanh` for
> `ReLU` and it is literally the curve-stacking picture from the talk.

> **📝 Gotcha (shapes).** The embedding gives a **3-D** tensor `(batch, block_size, emb_dim)` =
> `(batch, 3, 24)`. But `nn.Linear` wants a flat **2-D** input `(batch, features)`. So inside `forward`
> we **flatten** the last two axes into `3 × 24 = 72` numbers per example (`x.view(x.shape[0], -1)`).
> That flatten *is* the "concatenate the context characters" step — and it is the #1 shape bug here.

### 🎯 Task 2.1 & 2.2 — Define the network

Fill in the **two** TODOs in the class below: the **layers** in `__init__` (Task 2.1) and the
**forward pass** (Task 2.2). The shape comments on the right are your hints.

In [7]:
EMB_DIM  = 24     # numbers per character (more = better predictions; we PCA down to 2 to plot)
N_HIDDEN = 256    # size of the hidden layer

class NameMLP(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=EMB_DIM, n_hidden=N_HIDDEN):
        super().__init__()
        # 🎯 TODO (Task 2.1) — define the three layers: an embedding table and two linear layers
        self.emb = nn.Embedding(vocab_size, emb_dim)             # id -> (emb_dim,) vector
        self.fc1 = nn.Linear(block_size * emb_dim, n_hidden)     # hidden layer
        self.fc2 = nn.Linear(n_hidden, vocab_size)                # output layer (one score per character)

    def forward(self, x):                              # x: (batch, block_size) integer ids
        # 🎯 TODO (Task 2.2) — the forward pass: embed, flatten, hidden layer (tanh), output layer
        e = self.emb(x)                                # (batch, block_size, emb_dim)
        e = e.view(e.shape[0], -1)                      # flatten -> (batch, block_size * emb_dim)
        h = torch.tanh(self.fc1(e))                    # (batch, n_hidden)
        logits = self.fc2(h)                            # (batch, vocab_size)
        return logits

torch.manual_seed(SEED)
model = NameMLP(vocab_size, BLOCK_SIZE).to(device)
print(model)

# quick shape check on a small batch (moved to the device the model lives on)
logits = model(Xtr[:8].to(device))
n_params = sum(p.numel() for p in model.parameters())
print(f"\n{n_params} learnable parameters | logits shape: {tuple(logits.shape)}")
assert logits.shape == (8, vocab_size), f"expected (8, {vocab_size}), got {tuple(logits.shape)}"
print("✅ model defined — contexts turned into next-character scores")

NameMLP(
  (emb): Embedding(27, 24)
  (fc1): Linear(in_features=72, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=27, bias=True)
)

26275 learnable parameters | logits shape: (8, 27)
✅ model defined — contexts turned into next-character scores


## 3. The loss — how wrong is the model?

The network outputs a **score (logit)** for every possible next character. To train it we need one
number saying how wrong those scores are, given the *true* next character.

The standard choice for classification is **cross-entropy**, which combines **softmax** (turn scores
into probabilities) and the **negative log-likelihood** of the correct class. In PyTorch it is
`nn.CrossEntropyLoss()` — you hand it the **raw logits** (not softmaxed) and the **integer** target ids.

Sanity check: a freshly initialised model has learned nothing, so it should be about as confused as
random guessing over 27 characters — a loss of roughly `-ln(1/27) = ln(27) ≈ 3.30`.

### 🎯 Task 3.1 — Compute the loss

In [ ]:
loss_fn = nn.CrossEntropyLoss()

logits  = model(Xtr[:64].to(device))
targets = Ytr[:64].to(device)

# 🎯 TODO — apply loss_fn to the raw logits and the true next-character ids
loss = loss_fn(logits, targets)

print("loss on 64 examples  :", round(loss.item(), 4))
print("random-guess baseline:", round(float(np.log(vocab_size)), 4), "(= ln 27)")

assert loss.requires_grad, "loss must stay connected to the model (requires_grad=True)"
assert 2.5 < loss.item() < 4.5, "a fresh model should be roughly as confused as random guessing (~ln 27 = 3.3)"
print("✅ loss computed — now we can improve it")

> **📝 Gotcha.** Two classic mistakes `nn.CrossEntropyLoss` saves you from — as long as you feed it the
> right things:
> - **Do not softmax first.** It expects **raw logits** and applies softmax *internally*. Softmaxing
>   yourself applies it twice and quietly breaks training.
> - **Targets are integer class ids**, shape `(batch,)` — not one-hot vectors.

## 4. Training — the standard PyTorch loop

This is **lecture idea #2 in code: gradient descent.** Each step measures how wrong the model is (the
loss) and walks every parameter a little **downhill** on the loss surface — the "ball rolling down the
landscape" from the talk. Do it thousands of times and the model rolls into a good setting.

Concretely, it is the loop from the PyTorch tutorials. We wrap the data in a **`DataLoader`** that hands
us shuffled **minibatches**, and for every batch we run the same five steps:

1. **forward** — `pred = model(X)`
2. **loss** — `loss = loss_fn(pred, y)`
3. **zero_grad** — `optimizer.zero_grad()` (clear last step's gradients)
4. **backward** — `loss.backward()` (compute new gradients — which way is downhill?)
5. **step** — `optimizer.step()` (take one step downhill: nudge every parameter)

We call `model.train()` before training and `model.eval()` before evaluating, use the **`Adam`**
optimizer, and move each batch to the `device` with `.to(device)`.

> **💻 Runtime.** On a GPU runtime this trains in a few seconds; on CPU give it up to a minute.

> **📝 The #1 bug: `zero_grad`.** PyTorch **accumulates** gradients — every `.backward()` *adds* to
> whatever is already in `.grad`. If you forget to clear them each step (`optimizer.zero_grad()`), the
> gradients pile up and training diverges. Order matters: **zero_grad → backward → step**.

### 🎯 Task 4.1 — Fill in the five-step training loop

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 0.01

train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=BATCH_SIZE, shuffle=True)
optimizer    = torch.optim.Adam(model.parameters(), lr=LR)

total_steps = EPOCHS * len(train_loader)
log_every   = max(1, total_steps // 200)          # ~200 points on the curve, for any dataset size

steps_log, train_log, val_steps, val_log = [], [], [], []
step = 0
for epoch in range(EPOCHS):
    model.train()
    for Xb, Yb in train_loader:
        Xb, Yb = Xb.to(device), Yb.to(device)

        # 🎯 TODO — fill in the five steps of the training loop
        pred = ...                     # 1. forward:   contexts -> scores
        loss = ...             # 2. loss:      how wrong are we?
        ...                # 3. zero_grad: clear last step's gradients
        ...                      # 4. backward:  compute new gradients
        ...                     # 5. step:      update the parameters

        if step % log_every == 0:            # record the training loss for the curve
            steps_log.append(step); train_log.append(loss.item())
        step += 1

    # end of each epoch: evaluate on the held-out validation set
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xval.to(device)), Yval.to(device)).item()
    val_steps.append(step); val_log.append(val_loss)
    print(f"epoch {epoch + 1:2d}/{EPOCHS}   val loss {val_loss:.3f}   (latest train batch {train_log[-1]:.3f})")

assert train_log[-1] < train_log[0], "training loss should go DOWN as the model learns"
assert train_log[-1] < 3.0, "after training, the loss should be well below the ln(27) = 3.3 random baseline"
print(f"✅ the network learned — loss fell from ~{train_log[0]:.1f} (random guessing) toward ~{train_log[-1]:.1f}")

In [ ]:
#@title <👁️ Plot the training curve — Run Without Modification> {display-mode: "form"}
plt.figure(figsize=(7, 4))
plt.plot(steps_log, train_log, alpha=0.8, label="train (every step)")
plt.plot(val_steps, val_log, marker="o", color="tab:orange", label="validation (each epoch)")
plt.axhline(float(np.log(vocab_size)), color="gray", ls="--", label="random guessing (ln 27)")
plt.xlabel("training step"); plt.ylabel("cross-entropy loss"); plt.ylim(bottom=1.8)
plt.title("The model learns: loss falls from ~3.3 (random) toward ~2.2")
plt.legend(); plt.show()

> **🏁 Milestone.** You just trained a neural network with the standard PyTorch loop — the loss fell
> steeply as it learned how names are spelled. (On the full names dataset the **validation** loss falls
> right alongside the training loss, so the model is generalising, not merely memorising.)

## 5. Generate — invent brand-new names

Here is the key idea, worth pausing on: **the model never outputs a name.** For a given context it
outputs a **probability for every possible next character** — 27 numbers that sum to 1.

Generation is then a loop:

1. look at the probabilities the model gives for the next character,
2. **sample** one character from them (`torch.multinomial`),
3. add it to the context and repeat — stopping at the end token `.`.

Sampling (rather than always taking the single most-likely character) is what keeps the output varied.
First let's **see** those probabilities, then **sample** from them, then **watch a whole name** get built
one character at a time.

### 👁️ See the model think

Before generating anything, look at what the model actually produces. Type any context into
`context_text` below (try `q`, then `ma`, then `emm`) and see the probability it assigns to each of the
27 possible next characters. Notice how the distribution **changes with the context** — after `q` it is
almost certain the next letter is `u`. The model is not "drawing" a name; it is computing these numbers.

In [ ]:
#@title <👁️ Watch the model think: P(next character | context) — Run Without Modification> {display-mode: "form"}
context_text = "ma"  #@param {type:"string"}

context_text = context_text.lower()
ctx = ("." * BLOCK_SIZE + context_text)[-BLOCK_SIZE:]     # keep the last BLOCK_SIZE chars, pad with '.'
ids = torch.tensor([[stoi.get(c, 0) for c in ctx]], device=device)

model.eval()
with torch.no_grad():
    probs = F.softmax(model(ids), dim=-1).squeeze(0).cpu().numpy()

plt.figure(figsize=(9, 3))
plt.bar(range(vocab_size), probs, color="steelblue")
plt.xticks(range(vocab_size), [itos[i] for i in range(vocab_size)])
plt.ylabel("P(next char)")
plt.title(f"After context '{ctx}', the model's next-character probabilities")
plt.show()

print(f"context '{ctx}'  ->  top 5 next-character guesses:")
for i in probs.argsort()[::-1][:5]:
    label = itos[i] if i != 0 else ". (end of name)"
    print(f"   '{label}'   {probs[i] * 100:5.1f}%")

### 🎯 Task 5.1 — Sample the next character

Fill in the two lines that turn scores into a sampled character. Everything else — the sliding window,
the stop condition — is already wired up for you.

In [ ]:
@torch.no_grad()                                        # no gradients needed just to generate
def generate_name():
    model.eval()
    context = [0] * BLOCK_SIZE                           # start from '...'
    out = []
    for _ in range(20):                                 # safety cap on name length
        x = torch.tensor([context], device=device)       # shape (1, block_size)
        logits = model(x)                                 # (1, vocab_size)
        # 🎯 TODO — turn the logits into probabilities, then sample the next-character id
        probs = ...
        ix = ...
        if ix == 0:                                       # sampled the end token '.' -> stop
            break
        out.append(ix)
        context = context[1:] + [ix]                      # slide the window forward
    return decode(out)

print("one sample:", generate_name())
assert isinstance(generate_name(), str), "generate_name() should return a string"
print("✅ generation works")

### 👁️ Build a name, one character at a time

Now watch a whole name get built. Each row shows the current **context**, the model's **top guesses**,
the character it **picked** (sampled), and the **name so far** — so you can see it is genuinely built
from the context, not drawn whole. You can optionally **seed** it: type a few starting letters into
`seed` and watch the model continue them. (Notice the picked letter is not always the top guess — that
is sampling at work.)

In [ ]:
#@title <👁️ Build a name step by step (optionally seed it) — Run Without Modification> {display-mode: "form"}
seed = ""  #@param {type:"string"}

seed = seed.lower()
model.eval()
context = [0] * BLOCK_SIZE
name = ""
for c in seed:                                   # prime the context with the seed, if any
    context = context[1:] + [stoi.get(c, 0)]
    name += c

header = f"{'context':>8}   {'top guesses (prob)':<30}  picked   name so far"
print(header)
print("-" * len(header))
for _ in range(20):
    ids = torch.tensor([context], device=device)
    with torch.no_grad():
        probs = F.softmax(model(ids), dim=-1).squeeze(0)
    ix = int(torch.multinomial(probs, num_samples=1).item())
    p = probs.cpu().numpy()
    top = "  ".join(f"'{itos[j]}' {p[j] * 100:4.1f}%" for j in p.argsort()[::-1][:3])
    ctx_str = "".join(itos[i] for i in context)
    if ix == 0:
        print(f"{ctx_str:>8}   {top:<30}  {'. END':<7}  {name}")
        break
    name += itos[ix]
    print(f"{ctx_str:>8}   {top:<30}  {itos[ix]:<7}  {name}")
    context = context[1:] + [ix]

print(f"\nfinal name: {name}")

In [ ]:
#@title <👁️ Generate 20 brand-new names — Run Without Modification> {display-mode: "form"}
torch.manual_seed(SEED)   # so everyone sees the same names
names = [generate_name() for _ in range(20)]
print("names invented by the model:\n")
print("   " + "\n   ".join(names))

> **💡 You just built the core of an LLM.** "Predict the next token, then feed it back in" is *exactly*
> what ChatGPT does — only with sub-word **tokens** instead of characters, and a **transformer** instead
> of our little two-layer MLP. Everything else — embeddings, a forward pass, softmax, cross-entropy,
> gradient descent — is the same machinery you just wrote.

## 6. ⏭️ Optional — Peek inside: what did the model learn?

*(Bonus — skip if you are short on time; nothing below depends on it.)*

The model represents each character as a 24-number vector (the `nn.Embedding` weights). We can't plot
24 dimensions, so we compress them to 2 with **PCA** (principal component analysis — it keeps the
directions of biggest variation). Characters the model treats **similarly** end up **near each other**;
watch where the **vowels** land.

In [ ]:
#@title <👁️ Plot the learned character embeddings (PCA to 2-D) — Run Without Modification> {display-mode: "form"}
E = model.emb.weight.detach().cpu().numpy()     # nn.Embedding weights, shape (vocab_size, EMB_DIM)

# PCA via SVD: centre the vectors, then project onto the top 2 principal components
E = E - E.mean(axis=0)
U, S, Vt = np.linalg.svd(E, full_matrices=False)
coords = U[:, :2] * S[:2]                        # 2-D coordinates for each character

vowels = set("aeiou")
plt.figure(figsize=(7, 7))
for i in range(1, vocab_size):                   # skip the '.' boundary token (id 0)
    ch = itos[i]
    color = "crimson" if ch in vowels else "steelblue"
    plt.scatter(coords[i, 0], coords[i, 1], color=color, s=220)
    plt.text(coords[i, 0], coords[i, 1], ch, ha="center", va="center", color="white", fontsize=11)

plt.title("Learned character embeddings, PCA-projected to 2-D (red = vowels)")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.show()

> **💡 Learned representations.** Nobody programmed "vowel" into this model. It *invented* a geometry
> where similar characters sit together, purely from the task of predicting the next letter. Inspecting
> a model's internal representations like this — but for a **transformer**, with far more careful tooling
> — is the heart of **mechanistic interpretability**, Saturday's topic.
>
> **Honest caveat:** with only 2 dimensions and a small model, the structure is real but rough — do not
> over-read any single point, and expect the picture to shift a little if you retrain. Bigger models
> learn far richer representations.

## 7. ⏭️ Optional (advanced) — Level up: a WaveNet

*(Bonus — the "how do we do better?" answer. Skip unless you're curious; nothing depends on it.)*

Our MLP plateaus around **2.2** because it (a) sees only 3 characters and (b) squashes them together
all at once. The fix is a smarter **neural network** — no new ingredients, just better wiring. This is
Karpathy's **WaveNet** (makemore part 5), and it changes three things:

1. **Wider context** — it looks back **8** characters instead of 3.
2. **Hierarchical fusion** — instead of mashing all 8 together at once, it combines them **two at a time
   in a tree**: fuse (1,2),(3,4),(5,6),(7,8) → then fuse those pairs → then again. Structure the model
   can actually use.
3. **Batch normalization** (`nn.BatchNorm1d`) — a standard trick that normalises activations so a deeper
   network trains smoothly.

It is still embeddings + `nn.Linear` + `tanh` — the exact pieces you already used. Let's see it beat our MLP.

### The WaveNet — the flow of information

Instead of squashing all 8 characters together at once, it fuses them **two at a time, up a tree**
(8 → 4 → 2 → 1), so each step combines a bit more context:

```text
   c1 c2 c3 c4 c5 c6 c7 c8      8 characters (each embedded to 24 numbers)
      |
      v   fuse neighbouring PAIRS:  view -> Linear -> BatchNorm -> tanh
   [c1c2] [c3c4] [c5c6] [c7c8]   4 vectors
      |
      v   fuse neighbouring pairs again
   [ c1..c4 ]    [ c5..c8 ]      2 vectors
      |
      v   fuse the last pair
   [ c1......c8 ]                1 vector  (now "sees" all 8 characters)
      |
      v   Linear (-> 27 scores), then softmax
   P(next character)
```

In [ ]:
#@title <📊 Rebuild the pairs with an 8-character window — Run Without Modification> {display-mode: "form"}
WAVE_BLOCK = 8

def build_dataset_n(word_list, block):
    X, Y = [], []
    for w in word_list:
        ctx = [0] * block
        for ch in w + ".":
            ix = stoi[ch]; X.append(ctx); Y.append(ix); ctx = ctx[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

Xtr8, Ytr8   = build_dataset_n(train_words, WAVE_BLOCK)
Xval8, Yval8 = build_dataset_n(val_words, WAVE_BLOCK)
print("rebuilt with an 8-character window — Xtr8:", tuple(Xtr8.shape))

### 🎯 Tasks 7.1 & 7.2 — Build the WaveNet

Fill in the **two** TODOs in the class below (just like the MLP's Tasks 2.1 & 2.2):
- **Task 7.1** (`__init__`) — create the layers: an embedding, three `Linear + BatchNorm` fusion stages,
  and an output `Linear`.
- **Task 7.2** (`forward`) — the **hierarchical fusion**: combine the 8 characters two at a time, three
  times over (8 → 4 → 2 → 1), using `.view(batch, n, -1)` before each `Linear`. Follow the shape comments.

In [ ]:
class WaveNet(nn.Module):
    def __init__(self, vocab_size, emb_dim=24, n_hidden=256):
        super().__init__()
        # 🎯 TODO (Task 7.1) — define the layers: an embedding, three (Linear + BatchNorm) fusion
        #                       stages, and an output Linear
        self.emb = ...
        self.fc1 = ...;  self.bn1 = ...
        self.fc2 = ...; self.bn2 = ...
        self.fc3 = ...; self.bn3 = ...
        self.out = ...

    def _bn(self, layer, h):                  # apply BatchNorm over the feature axis of a (B, n, H) tensor
        B, n, H = h.shape
        return layer(h.reshape(B * n, H)).reshape(B, n, H)

    def forward(self, x):                                                     # x: (B, 8) integer ids
        e = self.emb(x)                                                       # (B, 8, emb_dim)
        # 🎯 TODO (Task 7.2) — fuse the 8 characters two-at-a-time, three times (match the shapes)
        h = ...  # fuse pairs -> (B, 4, H)
        h = ...  # fuse again -> (B, 2, H)
        h = ... # fuse again -> (B, H)
        return self.out(h)                                                   # (B, vocab_size)

torch.manual_seed(SEED)
wave = WaveNet(vocab_size).to(device)
print(wave)
print("parameters:", sum(p.numel() for p in wave.parameters()))

logits = wave(Xtr8[:8].to(device))
assert logits.shape == (8, vocab_size), f"expected (8, {vocab_size}), got {tuple(logits.shape)}"
print("✅ WaveNet forward pass works — the tree fuses 8 characters down to one prediction")

### Train the WaveNet

The punchline of the whole notebook: the **training loop does not change**. It is the exact same
five-step loop as Task 4.1 — only the model is different. Just run the cell below.

In [ ]:
#@title <📊 Train the WaveNet — Run Without Modification> {display-mode: "form"}
wave_loader = DataLoader(TensorDataset(Xtr8, Ytr8), batch_size=256, shuffle=True)
wave_opt    = torch.optim.Adam(wave.parameters(), lr=0.01)
WAVE_EPOCHS = 12

for epoch in range(WAVE_EPOCHS):
    wave.train()
    for Xb, Yb in wave_loader:               # the identical five-step loop from Task 4.1
        Xb, Yb = Xb.to(device), Yb.to(device)
        pred = wave(Xb)
        loss = loss_fn(pred, Yb)
        wave_opt.zero_grad()
        loss.backward()
        wave_opt.step()

    wave.eval()
    with torch.no_grad():
        wave_val = loss_fn(wave(Xval8.to(device)), Yval8.to(device)).item()
    print(f"epoch {epoch + 1:2d}/{WAVE_EPOCHS}   WaveNet val loss {wave_val:.3f}")

print(f"\nour MLP  val loss ~ {val_log[-1]:.3f}")
print(f"WaveNet  val loss ~ {wave_val:.3f}   (lower is better)")
print("✅ done — same training loop, a smarter model, a better score")

In [ ]:
#@title <👁️ Names invented by the WaveNet — Run Without Modification> {display-mode: "form"}
torch.manual_seed(SEED)
wave.eval()

def wave_generate():
    context = [0] * WAVE_BLOCK
    out = []
    for _ in range(20):
        with torch.no_grad():
            probs = F.softmax(wave(torch.tensor([context], device=device)), dim=-1)
        ix = int(torch.multinomial(probs, num_samples=1).item())
        if ix == 0:
            break
        out.append(ix); context = context[1:] + [ix]
    return decode(out)

print("names invented by the WaveNet:\n")
print("   " + "\n   ".join(wave_generate() for _ in range(20)))

### 👁️ Watch the WaveNet build a name, one character at a time

Same idea as the MLP's step-by-step view, now with the WaveNet's **8-character** context. Type a few
starting letters into `seed` and watch it continue them — try `ma`, `jul`, `chr`, or leave it blank.

In [ ]:
#@title <👁️ Build a name step by step with the WaveNet (optionally seed it) — Run Without Modification> {display-mode: "form"}
seed = ""  #@param {type:"string"}

seed = seed.lower()
wave.eval()
context = [0] * WAVE_BLOCK
name = ""
for c in seed:                                   # prime the 8-character context with the seed, if any
    context = context[1:] + [stoi.get(c, 0)]
    name += c

header = f"{'context':>10}   {'top guesses (prob)':<30}  picked   name so far"
print(header)
print("-" * len(header))
for _ in range(20):
    ids = torch.tensor([context], device=device)
    with torch.no_grad():
        probs = F.softmax(wave(ids), dim=-1).squeeze(0)
    ix = int(torch.multinomial(probs, num_samples=1).item())
    p = probs.cpu().numpy()
    top = "  ".join(f"'{itos[j]}' {p[j] * 100:4.1f}%" for j in p.argsort()[::-1][:3])
    ctx_str = "".join(itos[i] for i in context)
    if ix == 0:
        print(f"{ctx_str:>10}   {top:<30}  {'. END':<7}  {name}")
        break
    name += itos[ix]
    print(f"{ctx_str:>10}   {top:<30}  {itos[ix]:<7}  {name}")
    context = context[1:] + [ix]

print(f"\nfinal name: {name}")

> **💡 The ladder of models (all on the same names).** our MLP ~2.2 → this WaveNet ~2.0 → transformers
> (lower still). Every rung is *still a neural network* — the gains came from **architecture**, not magic:
> more context, used more cleverly. That last rung, **attention / transformers**, is Saturday's topic, and
> at the top sit the LLMs you use every day. Same game the whole way up: predict the next token, better and better.

## 🏁 Recap — and where this is going

You built a character-level neural language model, end to end, in idiomatic PyTorch:

- **Tokenization** — `text → integers`
- **Model** — an `nn.Module`: `nn.Embedding` → flatten → `nn.Linear` → tanh → `nn.Linear`
- **Loss** — `nn.CrossEntropyLoss`
- **Training** — the standard loop: `DataLoader`, `optimizer.zero_grad()` → `backward()` → `step()`
- **Generation** — autoregressive sampling of brand-new names
- **A peek inside** — the learned representations (vowels cluster)

**This is next-token prediction — the engine inside every LLM.** The rest of the weekend zooms into
that engine:

- You did **tokenization** by hand (character → integer); real LLMs automate it with sub-word tokenizers.
- You used a small **MLP** over a 3-character window. An LLM keeps the same task, loss, and sampling but
  swaps the MLP for a **transformer** so it can look much further back — that swap, **attention**, is what
  Saturday's *Mechanistic Interpretability* pries open.
- Tomorrow's *Prompt Optimization + GEPA* tunes the *text you feed* a frozen next-token model like this
  one — same machine, a different knob.
- And notice: the model *works*, but *why* it picked a given letter is already hard to read off its
  weights. That gap — capable models we cannot readily explain — is the reason this whole weekend on
  **explainable AI** exists.

See you in the next session.